In [1]:
import pandas as pd 
import duckdb

In [2]:
import plotly.express as px

In [2]:
conn=duckdb.connect(r"D:\SEC.gov project\database\credit_risk.db")

Schema Creation

In [83]:
conn.sql(""" 
         CREATE SCHEMA IF NOT EXISTS gold;
         """)

Metric Table Creation from NUM

In [134]:
conn.sql(""" 
         CREATE OR REPLACE TABLE gold.financial_metrics AS 
         SELECT DISTINCT
         adsh as accession_number,
         ddate as period_end
         FROM clean.num
         """)

Current Ratio

In [136]:
conn.sql("""
         ALTER TABLE gold.financial_metrics
         ADD COLUMN current_ratio DOUBLE;
         """)

CatalogException: Catalog Error: Column with name current_ratio already exists!

In [21]:
conn.sql("""
         UPDATE gold.financial_metrics g
         SET current_ratio = m.current_ratio
         FROM (
         WITH a AS(
             SELECT 
             adsh,
             ddate,
             MAX(CASE WHEN tag = 'AssetsCurrent' THEN value END) AS AssetsCurrent,
             MAX(CASE WHEN tag = 'LiabilitiesCurrent' THEN value END) AS LiabilitiesCurrent
             FROM clean.num
             WHERE tag IN ('AssetsCurrent', 'LiabilitiesCurrent')
             AND qtrs = 0
             GROUP BY adsh, ddate
             HAVING MAX(CASE WHEN tag = 'AssetsCurrent' THEN value END) IS NOT NULL
             AND MAX(CASE WHEN tag = 'LiabilitiesCurrent' THEN value END) IS NOT NULL
             )
        SELECT 
        adsh,
        ddate,
        ROUND(
        CASE WHEN AssetsCurrent>=0 AND LiabilitiesCurrent>=0 THEN AssetsCurrent/NULLIF(LiabilitiesCurrent, 0)
             WHEN AssetsCurrent>=0 AND LiabilitiesCurrent<=0 THEN AssetsCurrent/NULLIF(LiabilitiesCurrent, 0)
             WHEN AssetsCurrent<=0 AND LiabilitiesCurrent>=0 THEN AssetsCurrent/NULLIF(LiabilitiesCurrent, 0)
             WHEN AssetsCurrent<=0 AND LiabilitiesCurrent<=0 THEN AssetsCurrent/ABS(NULLIF(LiabilitiesCurrent , 0))
             ELSE NULL END, 2) AS current_ratio
        FROM a 
        )m
        WHERE g.accession_number = m.adsh
        AND g.period_end = m.ddate
        """)

Quick Ratio

In [140]:
conn.sql("""
         ALTER TABLE gold.financial_metrics
         ADD COLUMN quick_ratio DOUBLE;
         """)

In [22]:
conn.sql("""
         UPDATE gold.financial_metrics AS g 
         SET quick_ratio = m.quick_ratio
         FROM (
            WITH a AS (
               SELECT
                  adsh,
                  ddate,
                  MAX(CASE WHEN tag='AssetsCurrent' THEN value END) - 
                  MAX(CASE WHEN tag='InventoryNet' THEN value END)- 
                  MAX(CASE WHEN tag='PrepaidExpenseCurrent' THEN value END) AS quick_assets,
                  MAX(CASE WHEN tag='LiabilitiesCurrent' THEN value END) AS quick_liabilities
               FROM clean.num
               WHERE tag IN ('AssetsCurrent', 'InventoryNet', 'PrepaidExpenseCurrent', 'LiabilitiesCurrent')
               GROUP BY adsh, ddate
               HAVING MAX(CASE WHEN tag='AssetsCurrent' THEN value END) IS NOT NULL 
               AND MAX(CASE WHEN tag='LiabilitiesCurrent' THEN value END) IS NOT NULL 
               AND MAX(CASE WHEN tag='InventoryNet' THEN value END) IS NOT NULL 
               AND MAX(CASE WHEN tag='PrepaidExpenseCurrent' THEN value END) IS NOT NULL 
            )
            SELECT 
               adsh,
               ddate,
               ROUND(
                  CASE WHEN quick_assets>=0 AND quick_liabilities>=0 THEN quick_assets/NULLIF(quick_liabilities, 0)
                       WHEN quick_assets>=0 AND quick_liabilities<=0 THEN quick_assets/NULLIF(quick_liabilities, 0)
                       WHEN quick_assets<=0 AND quick_liabilities>=0 THEN quick_assets/NULLIF(quick_liabilities, 0)
                       WHEN quick_assets<=0 AND quick_liabilities<=0 THEN quick_assets/ABS(NULLIF(quick_liabilities, 0))
                       ELSE NULL END, 2) AS quick_ratio
            FROM a 
            )m
         WHERE g.accession_number = m.adsh
         AND g.period_end = m.ddate
            
         """)

Cash Ratio

In [142]:
conn.sql("ALTER TABLE gold.financial_metrics ADD COLUMN IF NOT EXISTS cash_ratio DOUBLE;")

In [23]:
conn.sql(""" 
         UPDATE gold.financial_metrics AS g
         SET cash_ratio = m.cash_ratio
         FROM (
            WITH a AS 
            (
                SELECT 
                    adsh,
                    ddate,
                    (
                        COALESCE(MAX(CASE WHEN tag = 'CashAndCashEquivalentsAtCarryingValue' THEN value END),
                                MAX(CASE WHEN tag = 'Cash' THEN value END), 0)
                        + COALESCE(
                            MAX(CASE WHEN tag = 'ShortTermInvestments' THEN value END),
                            MAX(CASE WHEN tag = 'MarketableSecuritiesCurrent' THEN value END),
                            COALESCE(MAX(CASE WHEN tag = 'AvailableForSaleSecuritiesCurrent' THEN value END), 0)
                            + COALESCE(MAX(CASE WHEN tag = 'TradingSecuritiesCurrent' THEN value END), 0),
                            0
                        )
                    ) AS cash_assets,
                    MAX(CASE WHEN tag = 'LiabilitiesCurrent' THEN value END) AS cash_liabilities
                FROM clean.num
                WHERE uom='USD' 
                AND tag IN (
                    'CashAndCashEquivalentsAtCarryingValue',
                    'Cash',
                    'ShortTermInvestments',
                    'MarketableSecuritiesCurrent',
                    'AvailableForSaleSecuritiesCurrent',
                    'TradingSecuritiesCurrent',
                    'LiabilitiesCurrent'
                )
                GROUP BY adsh, ddate
                HAVING MAX(CASE WHEN tag = 'LiabilitiesCurrent' THEN value END) IS NOT NULL
            )
            SELECT 
                adsh,
                ddate,
                ROUND(
                    CASE WHEN cash_assets>=0 AND cash_liabilities>=0 THEN cash_assets/NULLIF(cash_liabilities, 0)
                         WHEN cash_assets>=0 AND cash_liabilities<=0 THEN cash_assets/NULLIF(cash_liabilities, 0)
                         WHEN cash_assets<=0 AND cash_liabilities>=0 THEN cash_assets/NULLIF(cash_liabilities, 0)
                         WHEN cash_assets<=0 AND cash_liabilities<=0 THEN cash_assets/ABS(NULLIF(cash_liabilities, 0))
                         ELSE NULL END, 2) AS cash_ratio
            FROM a
            )m
            WHERE g.accession_number = m.adsh
            AND g.period_end = m.ddate
        """)

Net Working Capital

In [144]:
conn.sql("ALTER TABLE gold.financial_metrics ADD COLUMN net_working_capital DOUBLE")

In [24]:
conn.sql("""
         UPDATE gold.financial_metrics g 
         SET net_working_capital=m.net_working_capital
         FROM(
            SELECT 
            adsh,
            ddate,
            MAX(CASE WHEN tag='AssetsCurrent' THEN value END) -
            MAX(CASE WHEN tag='LiabilitiesCurrent' THEN value END) AS net_working_capital
            FROM clean.num 
            GROUP BY adsh,ddate
            HAVING MAX(CASE WHEN tag='AssetsCurrent' THEN value END) IS NOT NULL 
            AND MAX(CASE WHEN tag='LiabilitiesCurrent' THEN value END) IS NOT NULL 
         )m
         WHERE g.accession_number=m.adsh
         AND g.period_end=m.ddate;
         """)

Interest Coverage Ratio

In [146]:
conn.sql("ALTER TABLE gold.financial_metrics ADD COLUMN interest_coverage_ratio DOUBLE;")

In [25]:
conn.sql("""

UPDATE gold.financial_metrics AS g

SET interest_coverage_ratio = m.interest_coverage_ratio

FROM (

    WITH a AS (

        SELECT 
            n.adsh,
            n.ddate,

            COALESCE(
                MAX(CASE 
                    WHEN tag = 'OperatingIncomeLoss' 
                    THEN value 
                END),

                MAX(CASE 
                    WHEN tag = 'IncomeLossFromContinuingOperationsBeforeIncomeTaxesMinorityInterestAndIncomeTaxes' 
                    THEN value 
                END)
                +
                COALESCE(
                    MAX(CASE 
                        WHEN tag = 'InterestExpense' 
                        THEN value 
                    END),
                    0
                ),

                0
            ) AS EBIT,

            COALESCE(
                MAX(CASE 
                    WHEN tag = 'InterestExpense' 
                    THEN value 
                END),
                MAX(CASE 
                    WHEN tag = 'InterestExpenseDebt' 
                    THEN value 
                END),
                MAX(CASE 
                    WHEN tag = 'InterestAndDebtExpense' 
                    THEN value 
                END),
                MAX(CASE 
                    WHEN tag = 'InterestExpenseNet' 
                    THEN value 
                END)
            ) AS interest_expenses

        FROM clean.num n

        INNER JOIN clean.sub s 
            ON n.adsh = s.adsh

        WHERE n.uom = 'USD'

            AND n.tag IN (
                'OperatingIncomeLoss', 
                'IncomeLossFromContinuingOperationsBeforeIncomeTaxesMinorityInterestAndIncomeTaxes',
                'InterestExpense', 
                'InterestExpenseDebt', 
                'InterestAndDebtExpense', 
                'InterestExpenseNet'
            )

            AND (
                (s.fp IN ('Q1', 'Q2', 'Q3') AND n.qtrs = 1)
                OR 
                (s.fp = 'FY' AND n.qtrs = 4)
            )

        GROUP BY 
            n.adsh, 
            n.ddate

        HAVING MAX(
            CASE 
                WHEN tag IN (
                    'InterestExpense', 
                    'InterestExpenseDebt', 
                    'InterestAndDebtExpense', 
                    'InterestExpenseNet'
                )
                THEN value 
            END
        ) IS NOT NULL
    )

    SELECT 
        adsh,
        ddate,

        ROUND(
            CASE 
                WHEN EBIT >= 0 AND interest_expenses >= 0 
                    THEN EBIT / NULLIF(interest_expenses, 0)

                WHEN EBIT >= 0 AND interest_expenses <= 0 
                    THEN EBIT / NULLIF(interest_expenses, 0)

                WHEN EBIT <= 0 AND interest_expenses >= 0 
                    THEN EBIT / NULLIF(interest_expenses, 0)

                WHEN EBIT <= 0 AND interest_expenses <= 0 
                    THEN EBIT / ABS(NULLIF(interest_expenses, 0))

                ELSE NULL
            END,
            2
        ) AS interest_coverage_ratio

    FROM a

) AS m

WHERE g.accession_number = m.adsh
  AND g.period_end = m.ddate;

""")

OCF To Debt 

In [148]:
conn.sql(" ALTER TABLE gold.financial_metrics ADD COLUMN ocf_debt_ratio DOUBLE;")

In [26]:
conn.sql("""
UPDATE gold.financial_metrics AS g
SET ocf_debt_ratio = m.ocf_debt_ratio
FROM (
WITH a AS ( 
SELECT 
    adsh,
    ddate,
    COALESCE(
        MAX(CASE WHEN tag = 'NetCashProvidedByUsedInOperatingActivities' AND qtrs = 1 THEN value END),
        MAX(CASE WHEN tag = 'NetCashProvidedByUsedInOperatingActivitiesContinuingOperations' AND qtrs = 1 THEN value END)
    ) AS ocf,
    (
        COALESCE(MAX(CASE WHEN tag = 'ShortTermBorrowings' AND qtrs = 0 THEN value END), 0)
        +
        COALESCE(MAX(CASE WHEN tag = 'LongTermDebtCurrent' AND qtrs = 0 THEN value END), 0)
        +
        COALESCE(MAX(CASE WHEN tag = 'LongTermDebtNoncurrent' AND qtrs = 0 THEN value END), 0)
    ) AS debt
FROM clean.num
WHERE
    uom = 'USD'
    AND (
        ( tag IN (
            'NetCashProvidedByUsedInOperatingActivities', 
            'NetCashProvidedByUsedInOperatingActivitiesContinuingOperations')
            AND qtrs = 1
        )
        OR
        (
            tag IN (
                'ShortTermBorrowings',
                'LongTermDebtCurrent',
                'LongTermDebtNoncurrent'
            )
            AND qtrs = 0
        )
    )

GROUP BY 
    adsh, ddate
HAVING
    (
        COALESCE(MAX(CASE WHEN tag = 'ShortTermBorrowings' AND qtrs = 0 THEN value END), 0)
     +
        COALESCE(MAX(CASE WHEN tag = 'LongTermDebtCurrent' AND qtrs = 0 THEN value END), 0)
        +
        COALESCE(MAX(CASE WHEN tag = 'LongTermDebtNoncurrent' AND qtrs = 0 THEN value END), 0)
    ) > 0
    AND
    COALESCE(
        MAX(CASE WHEN tag = 'NetCashProvidedByUsedInOperatingActivities' AND qtrs = 1 THEN value END),
        MAX(CASE WHEN tag = 'NetCashProvidedByUsedInOperatingActivitiesContinuingOperations'AND qtrs = 1 THEN value END)
    ) IS NOT NULL 
)
SELECT 
adsh,
ddate,
ROUND(
        CASE 
            WHEN ocf >= 0 AND debt >= 0 
                THEN ocf / NULLIF(debt, 0)

            WHEN ocf >= 0 AND debt <= 0 
                THEN ocf / NULLIF(debt, 0)

            WHEN ocf <= 0 AND debt >= 0 
                THEN ocf / NULLIF(debt, 0)

            WHEN ocf <= 0 AND debt <= 0 
                THEN ocf / ABS(NULLIF(debt, 0))

            ELSE NULL
        END,
            2
    ) AS ocf_debt_ratio
FROM a 
)m
WHERE g.accession_number = m.adsh
AND g.period_end = m.ddate;
""")

Debt To Assets 

In [150]:
conn.sql("ALTER TABLE gold.financial_metrics ADD COLUMN IF NOT EXISTS debt_to_assets_ratio DOUBLE;")

In [27]:
conn.sql("""
UPDATE gold.financial_metrics AS g
SET debt_to_assets_ratio = m.debt_to_assets_ratio
FROM (
WITH a AS (
SELECT 
    adsh,
    ddate,
    -- NUMERATOR: Total Interest-Bearing Debt Summation
    (
        COALESCE(MAX(CASE WHEN tag = 'ShortTermBorrowings' THEN value END), 0) +
        COALESCE(MAX(CASE WHEN tag = 'LongTermDebtCurrent' THEN value END), 0) +
        COALESCE(MAX(CASE WHEN tag = 'LongTermDebtNoncurrent' THEN value END), 0) +
        COALESCE(MAX(CASE WHEN tag = 'FinanceLeaseLiabilityCurrent' THEN value END), 0) +
        COALESCE(MAX(CASE WHEN tag = 'FinanceLeaseLiabilityNoncurrent' THEN value END), 0)
    ) AS total_debt,
    
    -- DENOMINATOR: Total Assets Balance Sheet Anchor
    COALESCE(
        MAX(CASE WHEN tag = 'Assets' THEN value END),
        (COALESCE(MAX(CASE WHEN tag = 'AssetsCurrent' THEN value END), 0) + 
         COALESCE(MAX(CASE WHEN tag = 'AssetsNoncurrent' THEN value END), 0))
    ) AS total_assets
FROM clean.num
WHERE tag IN (
    'ShortTermBorrowings', 'LongTermDebtCurrent', 'LongTermDebtNoncurrent',
    'FinanceLeaseLiabilityCurrent', 'FinanceLeaseLiabilityNoncurrent',
    'Assets', 'AssetsCurrent', 'AssetsNoncurrent'
)
  AND qtrs = 0 
  AND uom = 'USD'
GROUP BY adsh, ddate
-- Structural Anchor: Only output if Total Assets is cleanly populated
HAVING MAX(CASE WHEN tag = 'Assets' THEN value END) IS NOT NULL 
    OR MAX(CASE WHEN tag = 'AssetsCurrent' THEN value END) IS NOT NULL
)
SELECT 
adsh,
ddate,
ROUND(
        CASE 
            WHEN total_debt >= 0 AND total_assets >= 0 
                THEN total_debt / NULLIF(total_assets, 0)

            WHEN total_debt >= 0 AND total_assets <= 0 
                THEN total_debt / NULLIF(total_assets, 0)

            WHEN total_debt <= 0 AND total_assets >= 0 
                THEN total_debt / NULLIF(total_assets, 0)

            WHEN total_debt <= 0 AND total_assets <= 0 
                THEN total_debt / ABS(NULLIF(total_assets, 0))

            ELSE NULL
        END, 2
    ) AS debt_to_assets_ratio
FROM a
) m
WHERE g.accession_number = m.adsh
AND g.period_end = m.ddate;
""")

Short Term Debt Ratio

In [152]:
conn.sql("ALTER TABLE gold.financial_metrics ADD COLUMN IF NOT EXISTS short_term_debt_ratio DOUBLE;")

In [34]:
conn.sql("""
UPDATE gold.financial_metrics AS g
SET short_term_debt_ratio = m.short_term_debt_ratio
FROM (
WITH debt AS (
    SELECT
        adsh,
        ddate,

        /* CURRENT DEBT */
        COALESCE(
            MAX(CASE WHEN tag = 'DebtCurrent' THEN value END),

            COALESCE(
                MAX(CASE WHEN tag = 'ShortTermBorrowings' THEN value END),
                0
            )
            +
            COALESCE(
                MAX(CASE
                    WHEN tag = 'LongTermDebtAndCapitalLeaseObligationsCurrent'
                    THEN value
                END),

                COALESCE(
                    MAX(CASE WHEN tag = 'LongTermDebtCurrent' THEN value END),
                    0
                )
                +
                COALESCE(
                    MAX(CASE WHEN tag = 'FinanceLeaseLiabilityCurrent' THEN value END),
                    MAX(CASE WHEN tag = 'CapitalLeaseObligationsCurrent' THEN value END),
                    0
                )
            )
        ) AS current_debt,

        /* NONCURRENT DEBT */
        COALESCE(
            MAX(CASE
                WHEN tag = 'LongTermDebtAndCapitalLeaseObligations'
                THEN value
            END),

            COALESCE(
                MAX(CASE WHEN tag = 'LongTermDebtNoncurrent' THEN value END),
                0
            )
            +
            COALESCE(
                MAX(CASE
                    WHEN tag = 'FinanceLeaseLiabilityNoncurrent'
                    THEN value
                END),
                MAX(CASE
                    WHEN tag = 'CapitalLeaseObligationsNoncurrent'
                    THEN value
                END),
                0
            )
        ) AS noncurrent_debt

    FROM clean.num

    WHERE
        qtrs = 0
        AND uom = 'USD'
        AND tag IN (
            'DebtCurrent',
            'ShortTermBorrowings',
            'LongTermDebtCurrent',
            'LongTermDebtAndCapitalLeaseObligationsCurrent',
            'FinanceLeaseLiabilityCurrent',
            'CapitalLeaseObligationsCurrent',
            'LongTermDebtAndCapitalLeaseObligations',
            'LongTermDebtNoncurrent',
            'FinanceLeaseLiabilityNoncurrent',
            'CapitalLeaseObligationsNoncurrent'
        )

    GROUP BY adsh, ddate
)

SELECT
    adsh,
    ddate,
    current_debt,
    noncurrent_debt,
    current_debt + noncurrent_debt AS total_debt,
    ROUND(
        CASE 
            WHEN current_debt >= 0 AND noncurrent_debt >= 0 
                    THEN current_debt / NULLIF(current_debt + noncurrent_debt, 0)
        
            WHEN current_debt >= 0 AND noncurrent_debt <= 0 
                    THEN current_debt / NULLIF(current_debt + noncurrent_debt, 0)
        
            WHEN current_debt <= 0 AND noncurrent_debt >= 0 
                    THEN current_debt / NULLIF(current_debt + noncurrent_debt, 0)
        
            WHEN current_debt <= 0 AND noncurrent_debt <= 0 
                    THEN current_debt / ABS(NULLIF(current_debt + noncurrent_debt, 0))
            ELSE NULL
        END,2
    ) AS short_term_debt_ratio

FROM debt
WHERE current_debt + noncurrent_debt > 0
)m
WHERE g.accession_number = m.adsh
AND g.period_end = m.ddate;
""")

ConnectionException: Connection Error: Connection already closed!

Cash Earnings Conversion

In [154]:
conn.sql("ALTER TABLE gold.financial_metrics ADD COLUMN IF NOT EXISTS cash_earnings_conversion DOUBLE;")

In [29]:
conn.sql("""
UPDATE gold.financial_metrics AS g
SET cash_earnings_conversion = m.cash_earnings_conversion
FROM (
WITH a AS (
SELECT 
    adsh,
    ddate,
    COALESCE(
        MAX(CASE WHEN tag = 'NetCashProvidedByUsedInOperatingActivities' THEN value END),
        MAX(CASE WHEN tag = 'NetCashProvidedByUsedInOperatingActivitiesContinuingOperations' THEN value END),
        0
    ) AS operating_cash_flow,
    COALESCE(
        MAX(CASE WHEN tag = 'NetIncomeLoss' THEN value END),
        MAX(CASE WHEN tag = 'ProfitLoss' THEN value END),
        MAX(CASE WHEN tag = 'NetIncomeLossAvailableToCommonStockholdersBasic' THEN value END),
        MAX(CASE WHEN tag = 'IncomeLossAttributableToParent' THEN value END),
        MAX(CASE WHEN tag = 'NetIncomeLossAllocatedToLimitedPartners' THEN value END),
        MAX(CASE WHEN tag = 'IncomeLossFromContinuingOperations' THEN value END)
    ) AS net_income

FROM clean.num
WHERE tag IN (
    'NetCashProvidedByUsedInOperatingActivities', 
    'NetCashProvidedByUsedInOperatingActivitiesContinuingOperations',
    'NetIncomeLoss', 
    'ProfitLoss', 
    'NetIncomeLossAvailableToCommonStockholdersBasic', 
    'IncomeLossAttributableToParent',
    'NetIncomeLossAllocatedToLimitedPartners',
    'IncomeLossFromContinuingOperations'
)
  AND qtrs = 1 
  AND uom = 'USD'
GROUP BY adsh, ddate
HAVING COALESCE(
    MAX(CASE WHEN tag = 'NetIncomeLoss' THEN value END),
    MAX(CASE WHEN tag = 'ProfitLoss' THEN value END),
    MAX(CASE WHEN tag = 'NetIncomeLossAvailableToCommonStockholdersBasic' THEN value END),
    MAX(CASE WHEN tag = 'IncomeLossAttributableToParent' THEN value END),
    MAX(CASE WHEN tag = 'NetIncomeLossAllocatedToLimitedPartners' THEN value END),
    MAX(CASE WHEN tag = 'IncomeLossFromContinuingOperations' THEN value END)
) IS NOT NULL
)
SELECT 
adsh,
ddate,
ROUND(
    CASE 
        WHEN operating_cash_flow >= 0 AND net_income >= 0 
                THEN operating_cash_flow / NULLIF(net_income, 0)
        
        WHEN operating_cash_flow >= 0 AND net_income <= 0 
                THEN operating_cash_flow / NULLIF(net_income, 0)
        
        WHEN operating_cash_flow <= 0 AND net_income >= 0 
                THEN operating_cash_flow / NULLIF(net_income, 0)
        
        WHEN operating_cash_flow <= 0 AND net_income <= 0 
                THEN operating_cash_flow / ABS(NULLIF(net_income, 0))
        ELSE NULL
    END,2
    ) AS cash_earnings_conversion
FROM a
) m
WHERE g.accession_number = m.adsh
AND g.period_end = m.ddate;
""")

FCF To Debt

In [156]:
conn.sql("ALTER TABLE gold.financial_metrics ADD COLUMN IF NOT EXISTS fcf_to_debt_ratio DOUBLE;")

In [30]:
conn.sql("""
UPDATE gold.financial_metrics AS g
SET fcf_to_debt_ratio = m.fcf_to_debt_ratio
FROM (
WITH compiled_fcf_debt AS (
    SELECT
        adsh,
        ddate,

        -- Quarterly Free Cash Flow = OCF - Capital Expenditures
        COALESCE(
            MAX(CASE
                WHEN tag = 'NetCashProvidedByUsedInOperatingActivities'
                AND qtrs = 1 THEN value
            END),
            MAX(CASE
                WHEN tag = 'NetCashProvidedByUsedInOperatingActivitiesContinuingOperations'
                AND qtrs = 1 THEN value
            END)
        )
        -
        ABS(
            COALESCE(
                MAX(CASE
                    WHEN tag = 'PaymentsToAcquirePropertyPlantAndEquipment'
                    AND qtrs = 1 THEN value
                END),
                MAX(CASE
                    WHEN tag = 'PaymentsToAcquireProductiveAssets'
                    AND qtrs = 1 THEN value
                END),
                MAX(CASE
                    WHEN tag = 'CapitalExpenditures'
                    AND qtrs = 1 THEN value
                END) 
            )
        ) AS quarterly_fcf,

        -- Current debt: aggregate tag first, then component fallback
        COALESCE(
            MAX(CASE
                WHEN tag = 'DebtCurrent'
                AND qtrs = 0 THEN value
            END),
            NULLIF(
                COALESCE(MAX(CASE
                    WHEN tag = 'ShortTermBorrowings'
                    AND qtrs = 0 THEN value
                END), 0)
                +
                COALESCE(MAX(CASE
                    WHEN tag = 'LongTermDebtCurrent'
                    AND qtrs = 0 THEN value
                END), 0)
                +
                COALESCE(MAX(CASE
                    WHEN tag = 'NotesPayableCurrent'
                    AND qtrs = 0 THEN value
                END), 0)
                +
                COALESCE(MAX(CASE
                    WHEN tag = 'FinanceLeaseLiabilityCurrent'
                    AND qtrs = 0 THEN value
                END), 0)
                +
                COALESCE(MAX(CASE
                    WHEN tag = 'CapitalLeaseObligationsCurrent'
                    AND qtrs = 0 THEN value
                END), 0),
                0
            ),
            MAX(CASE
                WHEN tag = 'LongTermDebtAndCapitalLeaseObligationsCurrent'
                AND qtrs = 0 THEN value
            END)
        ) AS current_debt,

        -- Non-current debt: aggregate tag first, then component fallback
        COALESCE(
            MAX(CASE
                WHEN tag = 'LongTermDebtAndCapitalLeaseObligationsNoncurrent'
                AND qtrs = 0 THEN value
            END),
            MAX(CASE
                WHEN tag = 'LongTermDebtAndCapitalLeaseObligations'
                AND qtrs = 0 THEN value
            END),
            NULLIF(
                COALESCE(MAX(CASE
                    WHEN tag = 'LongTermDebtNoncurrent'
                    AND qtrs = 0 THEN value
                END), 0)
                +
                COALESCE(MAX(CASE
                    WHEN tag = 'NotesPayableNoncurrent'
                    AND qtrs = 0 THEN value
                END), 0)
                +
                COALESCE(MAX(CASE
                    WHEN tag = 'FinanceLeaseLiabilityNoncurrent'
                    AND qtrs = 0 THEN value
                END), 0)
                +
                COALESCE(MAX(CASE
                    WHEN tag = 'CapitalLeaseObligationsNoncurrent'
                    AND qtrs = 0 THEN value
                END), 0),
                0
            )
        ) AS noncurrent_debt

    FROM clean.num

    WHERE
        (
            tag IN (
                'NetCashProvidedByUsedInOperatingActivities',
                'NetCashProvidedByUsedInOperatingActivitiesContinuingOperations',
                'PaymentsToAcquirePropertyPlantAndEquipment',
                'PaymentsToAcquireProductiveAssets',
                'CapitalExpenditures'
            )
            AND qtrs = 1
            AND uom = 'USD'
        )
        OR
        (
            tag IN (
                'DebtCurrent',
                'ShortTermBorrowings',
                'LongTermDebtCurrent',
                'NotesPayableCurrent',
                'FinanceLeaseLiabilityCurrent',
                'CapitalLeaseObligationsCurrent',
                'LongTermDebtAndCapitalLeaseObligationsCurrent',
                'LongTermDebtAndCapitalLeaseObligationsNoncurrent', -- Fixed: Included in filter array
                'LongTermDebtAndCapitalLeaseObligations',
                'LongTermDebtNoncurrent',
                'NotesPayableNoncurrent',
                'FinanceLeaseLiabilityNoncurrent',
                'CapitalLeaseObligationsNoncurrent'
            )
            AND qtrs = 0
            AND uom = 'USD'
        )

    GROUP BY adsh, ddate
)

SELECT
    adsh,
    ddate,
    quarterly_fcf,
    current_debt + noncurrent_debt AS total_debt,
    ROUND(
    CASE 
        WHEN quarterly_fcf >= 0 AND current_debt + noncurrent_debt >= 0 
                THEN quarterly_fcf / NULLIF(current_debt + noncurrent_debt, 0)
        
        WHEN quarterly_fcf >= 0 AND current_debt + noncurrent_debt <= 0 
                THEN quarterly_fcf / NULLIF(current_debt + noncurrent_debt, 0)
        
        WHEN quarterly_fcf <= 0 AND current_debt + noncurrent_debt >= 0 
                THEN quarterly_fcf / NULLIF(current_debt + noncurrent_debt, 0)
        
        WHEN quarterly_fcf <= 0 AND current_debt + noncurrent_debt <= 0 
                THEN quarterly_fcf / ABS(NULLIF(current_debt + noncurrent_debt, 0))
        ELSE NULL
    END,2
    ) AS fcf_to_debt_ratio

FROM compiled_fcf_debt

WHERE
    quarterly_fcf IS NOT NULL
    AND current_debt + noncurrent_debt > 0
) m
WHERE g.accession_number = m.adsh
AND g.period_end = m.ddate;
""")


Return On Assets

In [158]:
conn.sql("ALTER TABLE gold.financial_metrics ADD COLUMN IF NOT EXISTS return_on_assets DOUBLE;")

In [31]:
conn.sql("""
UPDATE gold.financial_metrics AS g
SET return_on_assets = m.return_on_assets
FROM (
WITH compiled_roa_components AS (
    SELECT
        adsh,
        ddate,

        -- 📉 NUMERATOR: "All-Weather" Discrete 3-Month Net Income (qtrs = 1)
        COALESCE(
            MAX(CASE WHEN tag = 'NetIncomeLoss' AND qtrs = 1 THEN value END),
            MAX(CASE WHEN tag = 'ProfitLoss' AND qtrs = 1 THEN value END),
            MAX(CASE WHEN tag = 'NetIncomeLossAvailableToCommonStockholdersBasic' AND qtrs = 1 THEN value END),
            MAX(CASE WHEN tag = 'IncomeLossAttributableToParent' AND qtrs = 1 THEN value END),
            MAX(CASE WHEN tag = 'NetIncomeLossAllocatedToLimitedPartners' AND qtrs = 1 THEN value END),
            MAX(CASE WHEN tag = 'IncomeLossFromContinuingOperations' AND qtrs = 1 THEN value END)
        ) AS quarterly_net_income,

        -- 🏢 DENOMINATOR: Snapshot Total Assets (qtrs = 0)
        COALESCE(
            MAX(CASE WHEN tag = 'Assets' AND qtrs = 0 THEN value END),
            CASE
            WHEN MAX(CASE WHEN tag = 'AssetsCurrent' AND qtrs = 0 THEN value END) IS NOT NULL
            AND MAX(CASE WHEN tag = 'AssetsNoncurrent' AND qtrs = 0 THEN value END) IS NOT NULL
            THEN MAX(CASE WHEN tag = 'AssetsCurrent' AND qtrs = 0 THEN value END)
            +
            MAX(CASE WHEN tag = 'AssetsNoncurrent' AND qtrs = 0 THEN value END)
            END
            ) AS total_assets

    FROM clean.num

    WHERE
        (
            tag IN (
                'NetIncomeLoss', 
                'ProfitLoss', 
                'NetIncomeLossAvailableToCommonStockholdersBasic', 
                'IncomeLossAttributableToParent',
                'NetIncomeLossAllocatedToLimitedPartners',
                'IncomeLossFromContinuingOperations'
            )
            AND qtrs = 1
            AND uom = 'USD'
        )
        OR
        (
            tag IN ('Assets', 'AssetsCurrent', 'AssetsNoncurrent')
            AND qtrs = 0
            AND uom = 'USD'
        )

    GROUP BY adsh, ddate
)

SELECT
    adsh,
    ddate,
    quarterly_net_income,
    total_assets,
    ROUND(
        CASE 
            WHEN quarterly_net_income >= 0 AND total_assets >= 0 
                    THEN quarterly_net_income / NULLIF(total_assets, 0)
            
            WHEN quarterly_net_income >= 0 AND total_assets <= 0 
                    THEN quarterly_net_income / NULLIF(total_assets, 0)
            
            WHEN quarterly_net_income <= 0 AND total_assets >= 0 
                    THEN quarterly_net_income / NULLIF(total_assets, 0)
            
            WHEN quarterly_net_income <= 0 AND total_assets <= 0 
                    THEN quarterly_net_income / ABS(NULLIF(total_assets, 0))
            ELSE NULL
        END,2
        ) AS return_on_assets

FROM compiled_roa_components

WHERE
    total_assets > 0
    AND quarterly_net_income IS NOT NULL
) m
WHERE g.accession_number = m.adsh
AND g.period_end = m.ddate;
""")

Consistency Validation

In [124]:
conn.sql("SELECT accession_number, period_end, count(*) FROM gold.financial_metrics group by accession_number, period_end having count(*) > 1 ").df()

,accession_number,period_end,count_star()


Complete NULL rows removed

In [32]:
#To count number of rows that have null values for all columns in gold.financial_metrics
conn.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE current_ratio IS NOT NULL
           OR quick_ratio IS NOT NULL
           OR cash_ratio IS NOT NULL
           OR net_working_capital IS NOT NULL
           OR interest_coverage_ratio IS NOT NULL
           OR ocf_debt_ratio IS NOT NULL
           OR debt_to_assets_ratio IS NOT NULL
           OR short_term_debt_ratio IS NOT NULL
           OR cash_earnings_conversion IS NOT NULL
           OR fcf_to_debt_ratio IS NOT NULL
           OR return_on_assets IS NOT NULL
    ) AS usable_rows
FROM gold.financial_metrics;
""").df()

,total_rows,usable_rows
0,10085,10075


In [ ]:
conn.close()